# PDF Exploration & Structure Analysis
## Industrial Equipment Data Extraction Project

This notebook systematically answers the following questions about the source PDFs:

| # | Question |
|---|----------|
| 1 | Are the PDFs text-based or scanned? |
| 2 | How many pages does each have? |
| 3 | What text can we extract? |
| 4 | Are there tables? |
| 5 | How is equipment information organized? |
| 6 | Are the two PDFs structured similarly? |
| 7 | What fields do we need to extract? |

**Source PDFs:**
- `AUSTCOLD.pdf`
- `MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf`

**Libraries used:** `pdfplumber`, `pypdfium2`, `pdfminer.six` (all pre-installed)


## 0. Setup & Imports

In [1]:
import os
import re
import sys
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import pdfplumber
import pypdfium2 as pdfium

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR     = PROJECT_ROOT / 'data' / 'raw'

PDF_AUSTCOLD = DATA_DIR / 'AUSTCOLD.pdf'
PDF_MYCOM    = DATA_DIR / 'MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf'

PDFS = {
    'AUSTCOLD': PDF_AUSTCOLD,
    'MYCOM':    PDF_MYCOM,
}

print('Library versions:')
import pdfplumber as _pp
print(f'  pdfplumber  : {_pp.__version__}')
#print(f'  pypdfium2   : {pdfium.__version__}')
print()
for name, path in PDFS.items():
    size_mb = path.stat().st_size / 1_048_576
    if path.exists():  
        print(f'{name}: {path.name}  ({size_mb:.1f} MB)')
    else:
        print(f'{name}: {path.name}  (file not found)')

Library versions:
  pdfplumber  : 0.11.10

AUSTCOLD: AUSTCOLD.pdf  (110.9 MB)
MYCOM: MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf  (47.5 MB)


## Are the PDFs text-based or scanned?

**Method:** We compare the number of extractable characters per page.
- **< 20 characters** on a page → treated as image-only (scanned)
- We use **pypdfium2** for fast per-page text extraction across the whole document.

In [2]:
TEXT_CHAR_THRESHOLD = 20  # chars/page below which we call it 'scanned'

def classify_pages(pdf_path: Path, sample_size: int = 30):
    """
    Open with pypdfium2, sample pages evenly, classify each as text or scanned.
    Returns: (total_pages, list[dict], overall_label)
    """
    doc = pdfium.PdfDocument(str(pdf_path))
    total_pages = len(doc)

    # Evenly spaced sample
    step = max(1, total_pages // sample_size)
    indices = list(range(0, total_pages, step))[:sample_size]

    results = []
    for i in indices:
        page = doc[i]
        textpage = page.get_textpage()
        text = textpage.get_text_range().strip()
        chars = len(text)
        results.append({
            'page':       i + 1,
            'chars':      chars,
            'is_scanned': chars < TEXT_CHAR_THRESHOLD,
        })
    doc.close()

    scanned_count = sum(r['is_scanned'] for r in results)
    ratio = scanned_count / len(results)
    if ratio > 0.5:
        overall = 'SCANNED'
    elif ratio > 0.1:
        overall = 'MIXED'
    else:
        overall = 'TEXT-BASED'
    return total_pages, results, overall, ratio


print('Classifying pages …\n')
print('=' * 65)
print(f"{'PDF':<12} {'Total Pages':>12} {'Sampled':>8} {'Scanned%':>9}  Classification")
print('=' * 65)

page_classifications = {}
for name, path in PDFS.items():
    total, page_data, overall, ratio = classify_pages(path)
    page_classifications[name] = (total, page_data, overall)
    print(f'{name:<12} {total:>12,} {len(page_data):>8} {ratio*100:>8.1f}%  {overall}')

print('=' * 65)

Classifying pages …

PDF           Total Pages  Sampled  Scanned%  Classification
AUSTCOLD            2,870       30     43.3%  MIXED
MYCOM               1,068       30     10.0%  TEXT-BASED


In [3]:
# Per-page detail table
for name, (total, page_data, overall) in page_classifications.items():
    print(f'  {name}:  {overall}  ({total:,} pages total)\n')
    print(f"  {'Page':>5}  {'Chars':>8}  Status")
    for r in page_data:
        icon = 'SCANNED' if r['is_scanned'] else 'TEXT'
        print(f"  {r['page']:>5}  {r['chars']:>8,}  {icon}")

    print()

  AUSTCOLD:  MIXED  (2,870 pages total)

   Page     Chars  Status
      1         0  SCANNED
     96         0  SCANNED
    191         0  SCANNED
    286         0  SCANNED
    381         0  SCANNED
    476         0  SCANNED
    571     2,541  TEXT
    666     2,697  TEXT
    761       564  TEXT
    856         0  SCANNED
    951     1,397  TEXT
   1046       737  TEXT
   1141     1,514  TEXT
   1236       148  TEXT
   1331       598  TEXT
   1426     1,549  TEXT
   1521       152  TEXT
   1616         0  SCANNED
   1711     1,785  TEXT
   1806     2,020  TEXT
   1901         0  SCANNED
   1996     1,468  TEXT
   2091     6,782  TEXT
   2186         0  SCANNED
   2281     2,043  TEXT
   2376         0  SCANNED
   2471     2,153  TEXT
   2566         0  SCANNED
   2661       913  TEXT
   2756         0  SCANNED

  MYCOM:  TEXT-BASED  (1,068 pages total)

   Page     Chars  Status
      1       491  TEXT
     36         0  SCANNED
     71       382  TEXT
    106       380  TEXT
    1


## What text can we extract?

We sample pages from the beginning, middle, and end of each document and inspect the raw extracted text.

For a more readable result, we used the (x, y) positions of the characters to preserve their spatial layout when printing the extracted text.

In [4]:
from pathlib import Path
import pypdfium2 as pdfium


def extract_words_with_positions(pdf_path: Path, page_index: int):
    """Extrait les mots avec leur position sur la page."""

    doc = pdfium.PdfDocument(str(pdf_path))

    if not (0 <= page_index < len(doc)):
        doc.close()
        return []

    page = doc[page_index]
    text_page = page.get_textpage()

    n_chars = text_page.count_chars()

    chars = []

    for i in range(n_chars):

        char = text_page.get_text_range(i, 1)

        if not char:
            continue

        try:
            x0, y0, x1, y1 = text_page.get_charbox(i)
        except Exception:
            continue

        chars.append({
            "char": char,
            "x0": x0,
            "y0": y0,
            "x1": x1,
            "y1": y1
        })

    # REGROUPER LES CARACTÈRES EN MOTS

    words = []

    current_word = []

    for c in chars:

        if c["char"].isspace():

            if current_word:

                words.append({
                    "text": "".join(x["char"] for x in current_word),
                    "x0": min(x["x0"] for x in current_word),
                    "y0": min(x["y0"] for x in current_word),
                    "x1": max(x["x1"] for x in current_word),
                    "y1": max(x["y1"] for x in current_word)
                })

                current_word = []

        else:
            current_word.append(c)

    # Dernier mot
    if current_word:

        words.append({
            "text": "".join(x["char"] for x in current_word),
            "x0": min(x["x0"] for x in current_word),
            "y0": min(x["y0"] for x in current_word),
            "x1": max(x["x1"] for x in current_word),
            "y1": max(x["y1"] for x in current_word)
        })

    text_page.close()
    page.close()
    doc.close()

    return words



# PRINT BASÉ SUR LA LOCALISATION

def print_page_using_positions(words, page_width=100):
    """
    Affiche le texte en utilisant les coordonnées X/Y
    des mots afin de reproduire approximativement
    la disposition spatiale de la page.
    """

    if not words:
        print("[Aucun texte]")
        return

    
    # 1. Déterminer les limites de la page
    
    min_x = min(word["x0"] for word in words)
    max_x = max(word["x1"] for word in words)

    min_y = min(word["y0"] for word in words)
    max_y = max(word["y1"] for word in words)

    width = max_x - min_x
    height = max_y - min_y

   
    # 2. Regrouper les mots par ligne selon Y
    
    words_sorted = sorted(
        words,
        key=lambda w: -w["y1"]
    )

    lines = []

    # Tolérance verticale
    y_tolerance = 4

    for word in words_sorted:

        word_y = (word["y0"] + word["y1"]) / 2

        found_line = None

        for line in lines:

            if abs(word_y - line["y"]) <= y_tolerance:
                found_line = line
                break

        if found_line is None:

            lines.append({
                "y": word_y,
                "words": [word]
            })

        else:

            found_line["words"].append(word)

    
    # 3. Trier les lignes verticalement
   
    lines.sort(
        key=lambda line: -line["y"]
    )
 
    # 4. Afficher chaque ligne selon X
    
    previous_y = None

    for line in lines:

        line["words"].sort(
            key=lambda w: w["x0"]
        )
        # Déterminer l'écart vertical entre les lignes
        if previous_y is not None:

            y_gap = abs(previous_y - line["y"])

            # Ajouter une ligne vide si grand espace vertical
            if height > 0 and y_gap > height * 0.04:
                print()

        previous_y = line["y"]
        current_position = 0
        output = ""

        for word in line["words"]:
            # Position X normalisée entre 0 et page_width
            if width > 0:

                x_position = int(
                    ((word["x0"] - min_x) / width)
                    * page_width
                )

            else:
                x_position = 0

            # Nombre d'espaces nécessaires
            spaces = max(
                1,
                x_position - current_position
            )

            output += " " * spaces
            output += word["text"]

            current_position = (
                x_position + len(word["text"])
            )

        print(output.rstrip())

# EXÉCUTION

TARGET_PAGE_INDEX = 0

for name, path in PDFS.items():

    words = extract_words_with_positions(
        path,
        TARGET_PAGE_INDEX
    )

    page_number = TARGET_PAGE_INDEX + 1

    
    print(
        f"{name} — Page {page_number}"
    )
    

    # Le print utilise directement la localisation X/Y
    print_page_using_positions(words)


AUSTCOLD — Page 1
[Aucun texte]
MYCOM — Page 1
                        CLIENT:     OCP - MAROC PHOSPHORE             Vendor Job.N°:      2012-123
                        PROJ.N°:    Q3600XXX                         Unit:                455A
                        PLANT:      ODI at P1 SITE                      OCP Doc. Code:  455A-REF-PRD- 009
                                    N°2 AMMONIA STORAGE TANKS          Vendor Doc.No.: P1-REF-PRD-12-123-009
                        LOCATION:   JORF LASFAR - MOROCCO              Sheet           1    of    1068
                                                                     REV.                  0

                                 OCP  - MAROC    PHOSPHORE

                 OPERATING            AND     MAINTENANCE              DATA

                                                             This  document   contains
                                                             Mayekawa's   document   n°
                                   

In [5]:
# Text quality metrics across the first 60 pages
def text_quality_metrics(pdf_path: Path, max_pages: int = 60) -> dict:
    doc = pdfium.PdfDocument(str(pdf_path))
    n = min(len(doc), max_pages)
    total_chars, total_words, blank_pages, text_pages = 0, 0, 0, 0

    for i in range(n):
        txt = doc[i].get_textpage().get_text_range().strip()
        c = len(txt)
        w = len(txt.split())
        total_chars += c
        total_words += w
        if c < TEXT_CHAR_THRESHOLD:
            blank_pages += 1
        else:
            text_pages += 1
    doc.close()

    return {
        'pages_scanned':              n,
        'text_pages':                 text_pages,
        'blank_or_image_pages':       blank_pages,
        'avg_chars_per_text_page':    round(total_chars / max(text_pages, 1)),
        'avg_words_per_text_page':    round(total_words / max(text_pages, 1)),
        'total_words_sampled':        total_words,
        'total_chars_sampled':        total_chars,
    }


print('Text Quality Metrics  (first 60 pages of each PDF)')
print()
for name, path in PDFS.items():
    m = text_quality_metrics(path)
    print(f'\n  {name}')
    for k, v in m.items():
        print(f'    {k:<35} {v:>10,}')

Text Quality Metrics  (first 60 pages of each PDF)


  AUSTCOLD
    pages_scanned                               60
    text_pages                                  39
    blank_or_image_pages                        21
    avg_chars_per_text_page                  1,589
    avg_words_per_text_page                    235
    total_words_sampled                      9,182
    total_chars_sampled                     61,957

  MYCOM
    pages_scanned                               60
    text_pages                                  37
    blank_or_image_pages                        23
    avg_chars_per_text_page                    755
    avg_words_per_text_page                    106
    total_words_sampled                      3,923
    total_chars_sampled                     27,940



## Are there tables?

**pdfplumber** is the best open-source tool for detecting tables in text-based PDFs — it analyses line geometry to identify cell boundaries.

In [6]:
def find_tables(pdf_path: Path, max_pages: int = 80) -> list[dict]:
    """Use pdfplumber to detect tables and return metadata + sample rows."""
    found = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        n = min(len(pdf.pages), max_pages)
        for i in range(n):
            tables = pdf.pages[i].extract_tables()
            for t_idx, table in enumerate(tables):
                # Filter empty rows
                rows = [r for r in table if any(c for c in r if c and str(c).strip())]
                if len(rows) >= 2:
                    found.append({
                        'page':         i + 1,
                        'table_index':  t_idx,
                        'rows':         len(rows),
                        'cols':         max(len(r) for r in rows),
                        'header_guess': rows[0],
                        'sample_row':   rows[1] if len(rows) > 1 else [],
                        'all_rows':     rows,
                    })
    return found


print('Scanning for tables (first 80 pages) …\n')
all_tables = {}
for name, path in PDFS.items():
    tables = find_tables(path)
    all_tables[name] = tables
    total, _, _ = page_classifications[name]
    print(f'  {name}: {len(tables)} table(s) found  (scanned {min(total,80)} of {total} pages)')

Scanning for tables (first 80 pages) …

  AUSTCOLD: 18 table(s) found  (scanned 80 of 2870 pages)
  MYCOM: 65 table(s) found  (scanned 80 of 1068 pages)


In [7]:
MAX_TABLES_TO_SHOW = 12

for name, tables in all_tables.items():
    print(f"\n{'━'*65}")
    print(f'  {name}  —  Table Inventory  ({len(tables)} tables)')
    print(f"{'━'*65}")
    if not tables:
        print('  (no tables detected in the first 80 pages)')
    for i, t in enumerate(tables[:MAX_TABLES_TO_SHOW]):
        header = [str(h)[:28] for h in t['header_guess']]
        sample = [str(v)[:28] for v in t['sample_row']]
        print(f"\n  Table {i+1}  (page {t['page']}, {t['rows']} rows × {t['cols']} cols)")
        print(f'    Header : {header}')
        print(f'    Row[1] : {sample}')
    if len(tables) > MAX_TABLES_TO_SHOW:
        print(f'\n  … and {len(tables) - MAX_TABLES_TO_SHOW} more tables not shown.')


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  AUSTCOLD  —  Table Inventory  (18 tables)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Table 1  (page 28, 8 rows × 6 cols)
    Header : ['AUSTCOLD REFRIGERATION PTY. ', 'None', 'None', 'None', 'None', 'None']
    Row[1] : ["TITRE :\nJOURNAL DE L'HISTORI", 'None', 'None', 'None', 'None', 'PAGE']

  Table 2  (page 31, 8 rows × 6 cols)
    Header : ['AUSTCOLD REFRIGERATION PTY. ', 'None', 'None', 'None', 'None', 'None']
    Row[1] : ['TITLE :\nPLANT HISTORY LOG', 'None', 'None', 'None', 'None', 'PAGE']

  Table 3  (page 34, 8 rows × 11 cols)
    Header : ['AUSTCOLD REFRIGERATION PTY. ', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None']
    Row[1] : ['TITLE :\nOIL CONSUMPTION LOG', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'PAGE']

  Table 4  (page 35, 8 rows × 11 cols)
    Header : ['AUSTCOLD REFRIGERATION PTY. ', 'None', 'None', 'None', 


## How is equipment information organized?

We scan for:
- **Section headings** (numbered titles)
- **Key-value pairs** (label: value)
- **Engineering units** (kW, RPM, kPa, °C …)
- **Domain-specific signals** (Equipment name, reference, serial number, power)

In [8]:
PATTERNS = {
    'Numeric + Unit': re.compile(
        r'\b(\d+\.?\d*)\s*'
        r'(kW|kPa|bar|RPM|Hz|V|A|°C|°F|kg|L|cfm|m³|psig|psi|hp|kJ|MPa)\b',
        re.I
    ),

    # Plus strict
    'Section Heading': re.compile(
        r'^\s*'
        r'(\d{1,2}(?:\.\d{1,3}){0,3})'
        r'\s+'
        r'([A-Z][A-Za-zÀ-ÿ0-9][^\n]{5,70})'
        r'\s*$',
        re.M
    ),

    'Key-Value pair': re.compile(
        r'^\s*([A-Za-z][\w\s/()]{2,40})\s*[:\-]\s*(.{1,80})$',
        re.M
    ),

    "Model": re.compile(
        r'\b(?:model|modèle|type)\s*(?:no\.?|number|n°|#)?'
        r'\s*[:\-]?\s*([A-Z0-9][A-Z0-9._/\-]{2,})',
        re.I
    ),

    "Reference": re.compile(
        r'\b(?:reference|référence|ref\.?|ref)'
        r'\s*[:#\-]?\s*([A-Z0-9][A-Z0-9._/\-]{2,})',
        re.I
    ),

    "Power": re.compile(
        r'\b(?:power|puissance|rating|rated\s*power|'
        r'motor\s*power|puissance\s*moteur)'
        r'\s*[:#=\-]?\s*'
        r'(-?\d+(?:[.,]\d+)?)\s*'
        r'(kW|W|MW|hp|CV)\b',
        re.I
    ),
}


In [9]:
def is_valid_section_heading(text):
    """
    Vérifie qu'une ligne détectée comme heading
    ressemble réellement à un titre de section.
    """

    text = text.strip()

    # Trop court
    if len(text) < 8:
        return False

    # Trop long pour être un heading
    if len(text) > 80:
        return False

    # Exclure les métadonnées / références

    metadata_patterns = [
        r'^\s*doc\.?\s*no\.?',
        r'^\s*document\s*(no|number)',
        r'^\s*ref\.?',
        r'^\s*reference',
        r'^\s*model\s*(no|number)?',
        r'^\s*serial\s*(no|number)?',
        r'^\s*s/?n\b',
        r'^\s*part\s*(no|number)?',
        r'^\s*item\s*(no|number)?',
        r'^\s*drawing\s*(no|number)?',
        r'^\s*order\s*(no|number)?',
        r'^\s*project\s*(no|number)?',
        r'^\s*customer\s*(no|number)?',
        r'^\s*tag\s*(no|number)?',
    ]

    for pattern in metadata_patterns:
        if re.search(pattern, text, re.I):
            return False

    # Exclure certaines descriptions techniques

    technical_phrases = [
        r'\bgreased\b',
        r'\bfactory\b',
        r'\bvoltage\b',
        r'\bpower\b',
        r'\bpressure\b',
        r'\btemperature\b',
        r'\bcapacity\b',
        r'\brating\b',
        r'\bspeed\b',
        r'\bfrequency\b',
        r'\bweight\b',
        r'\bdimension\b',
        r'\bmaterial\b',
        r'\bserial\b',
        r'\bmodel\b',
    ]

    for pattern in technical_phrases:
        if re.search(pattern, text, re.I):
            return False

    # Exclure les lignes contenant principalement des chiffres

    digits = sum(c.isdigit() for c in text)
    letters = sum(c.isalpha() for c in text)

    if letters == 0:
        return False

    # Trop de chiffres par rapport aux lettres
    if digits > letters:
        return False

    # Exclure les références du type ABC-123-XYZ

    if re.search(
        r'\b[A-Z]{2,}[-_/]?\d{2,}[A-Z0-9/_-]*\b',
        text
    ):
        return False

    # Exclure les lignes avec unités

    units = [
        'kW', 'kPa', 'bar', 'RPM', 'Hz',
        'V', 'A', '°C', '°F',
        'kg', 'L', 'cfm', 'psi',
        'psig', 'MPa', 'hp'
    ]

    for unit in units:
        if re.search(rf'\b{re.escape(unit)}\b', text, re.I):
            return False

    return True


In [10]:
def analyze_structure(pdf_path: Path, max_pages: int = 60):
    """
    Extract pattern hits and full text from up to max_pages pages.

    Section headings are additionally filtered to remove
    metadata, references, technical descriptions and values.
    """

    pattern_hits = defaultdict(list)
    full_text = ''

    doc = pdfium.PdfDocument(str(pdf_path))

    n = min(len(doc), max_pages)

    for i in range(n):

        page_txt = doc[i].get_textpage().get_text_range()

        full_text += page_txt

        # SEARCH ALL PATTERNS

        for label, pat in PATTERNS.items():

            for m in pat.finditer(page_txt):

                match_text = m.group(0).strip()

                # SPECIAL FILTER FOR SECTION HEADINGS

                if label == 'Section Heading':

                    if not is_valid_section_heading(match_text):
                        continue

                pattern_hits[label].append({
                    'page': i + 1,
                    'match': match_text[:80]
                })

    doc.close()

    # ============================================================
    # DE-DUPLICATE SECTION HEADINGS
    # ============================================================

    seen_headings = set()
    unique_headings = []

    for h in pattern_hits['Section Heading']:

        t = h['match'].strip()

        if t not in seen_headings:

            unique_headings.append(h)
            seen_headings.add(t)

    return pattern_hits, unique_headings, full_text


print('Analyzing document structure (first 60 pages each) …')
doc_analysis = {}
for name, path in PDFS.items():
    hits, headings, full_text = analyze_structure(path)
    doc_analysis[name] = (hits, headings, full_text)
    print(f'  {name}')
print('Done.')

Analyzing document structure (first 60 pages each) …
  AUSTCOLD
  MYCOM
Done.


In [11]:
# Pattern hit summary table
print('Pattern Hit Summary  (first 60 pages per PDF)')
print()
header_line = f"  {'Pattern':<22}" + ''.join(f'  {n:>15}' for n in PDFS)
print(header_line)
print('  ' + '─' * (20 + 17 * len(PDFS)))

for label in PATTERNS:
    row = f'  {label:<22}'
    for name in PDFS:
        hits, _, _ = doc_analysis[name]
        row += f'  {len(hits[label]):>15,}'
    print(row)

Pattern Hit Summary  (first 60 pages per PDF)

  Pattern                        AUSTCOLD            MYCOM
  ──────────────────────────────────────────────────────
  Numeric + Unit                      227               75
  Section Heading                     144               44
  Key-Value pair                      121              235
  Model                                 0               18
  Reference                            76              103
  Power                                 0                0


In [12]:
# Section headings (table-of-contents proxy)
MAX_HEADINGS = 40

for name in PDFS:
    _, headings, _ = doc_analysis[name]
    print(f"\n{'━'*60}")
    print(f' {name}  —  Detected Section Headings  ({len(headings)} unique)')
    print(f"{'━'*60}")
    if not headings:
        print('  (none — headings may not follow a numbered format)')
    for h in headings[:MAX_HEADINGS]:
        print(f"    p.{h['page']:>4}   {h['match'].strip()}")
    if len(headings) > MAX_HEADINGS:
        print(f'    … and {len(headings)-MAX_HEADINGS} more')


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 AUSTCOLD  —  Detected Section Headings  (144 unique)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    p.   3   1.01 Contact details
    p.   3   1.03 Safety precautions
    p.   3   1.04 Package warranty and support information
    p.   3   1.05 Plant history log
    p.   3   1.06 Oil consumption log
    p.   3   1.07 Utilities consumption schedule
    p.   3   1.08 Cause & effect chart
    p.   3   2.01 Control Philosophy
    p.   3   3.01 Technical schedule
    p.   3   3.02 Compressor data sheet
    p.   3   3.03 Compressor motor data sheet
    p.   3   3.04 Oil pump data sheet
    p.   3   3.05 Oil pump motor data sheet
    p.   3   3.06 Oil cooler data sheet
    p.   3   3.07 Oil separator data sheet
    p.   3   3.08 Liquid receiver data sheet
    p.   3   3.09 Purger data sheet
    p.   3   3.10 Economiser data sheet
    p.   3   3.11 Condenser data sheet
    p.   3   3.12 Condenser fan data sheet
   

Section-heading extraction is not required for the main information extraction or summarization task. The document summary provides the necessary high-level understanding of the content.

However, we keep heading extraction as an additional **structural and navigation tool**. This is particularly useful for documents where the table of contents or summary page lists the sections but does **not provide associated page numbers**.

In [13]:
# Equipment signal samples: top 6 matches per signal type
SHOW = 6
SIGNAL_LABELS = ['Model', 'Reference', 'Power', 'Numeric + Unit']

    

for name in PDFS:
    hits, _, _ = doc_analysis[name]
    print(f"\n")
    print(f'  {name}  —  Equipment Signal Samples')
    for label in SIGNAL_LABELS:
        items = hits[label][:SHOW]
        total = len(hits[label])
        if items:
            print(f'\n  [{label}]  ({total} total hits)')
            for item in items:
                print(f"    p.{item['page']:>3}  {item['match']}")



  AUSTCOLD  —  Equipment Signal Samples

  [Reference]  (76 total hits)
    p.  4  Refrigeration
    p.  6  refroidisseur
    p.  7  refroidisseur
    p. 11  REFRIGERATION
    p. 11  REFERENCE : A5114
    p. 14  REFRIGERATION

  [Numeric + Unit]  (227 total hits)
    p. 18  1.1 A
    p. 39  50Hz
    p. 39  100A
    p. 39  1 kW
    p. 39  100A
    p. 39  50Hz


  MYCOM  —  Equipment Signal Samples

  [Model]  (18 total hits)
    p. 24  Type: Horizontal
    p. 24  Type Plug
    p. 24  Type Extruded
    p. 24  Model: 1829-4-19L/22MT
    p. 24  Type: V-Belt
    p. 33  Type BEM

  [Reference]  (103 total hits)
    p.  1  REF-PRD-
    p.  1  REF-PRD-12-123-009
    p.  2  REF-PRD-
    p.  2  REF-PRD-12-123-009
    p.  3  ref: 12OP0003
    p.  3  REFRIGERATION

  [Numeric + Unit]  (75 total hits)
    p.  1  455A
    p.  1  455A
    p.  2  455A
    p.  2  455A
    p.  4  455A
    p.  4  455A


In [14]:
# ── 7b. Domain-driven: curated extraction schema ─────────────────────────
EXTRACTION_SCHEMA = {
    'Identification': [
        ('model_number',        'Model or product number of the unit'),
        ('serial_number',       'Unique serial number'),
        ('item_number',         'Item or tag number'),
        ('part_number',         'Spare part identifier'),
        ('equipment_tag',       'Plant / P&ID tag'),
        ('manufacturer',        'OEM name'),
        ('document_title',      'Title of the manual / datasheet'),
        ('document_revision',   'Revision or version code'),
        ('document_date',       'Issue or revision date'),
    ],
    'Refrigeration Circuit': [
        ('refrigerant_type',         'Refrigerant designation e.g. R-717, R-404A'),
        ('refrigerant_charge_kg',    'Total refrigerant charge in kg'),
        ('compressor_type',          'Screw / reciprocating / scroll / centrifugal'),
        ('compressor_model',         'Compressor model number'),
        ('cooling_capacity_kW',      'Net cooling capacity in kW'),
        ('suction_temp_degC',        'Suction / evaporation temperature °C'),
        ('discharge_temp_degC',      'Discharge / condensing temperature °C'),
        ('suction_pressure_kPa',     'Low-side pressure kPa(g)'),
        ('discharge_pressure_kPa',   'High-side pressure kPa(g)'),
        ('operating_speed_RPM',      'Compressor shaft speed RPM'),
        ('COP',                      'Coefficient of performance'),
    ],
    'Electrical': [
        ('supply_voltage_V',        'Supply voltage e.g. 415 V'),
        ('frequency_Hz',            'Supply frequency Hz'),
        ('power_input_kW',          'Total electrical power input kW'),
        ('full_load_current_A',     'FLA in amperes'),
        ('motor_model',             'Motor model number'),
        ('motor_frame',             'IEC / NEMA frame size'),
        ('IP_rating',               'Ingress protection class'),
    ],
    'Physical': [
        ('weight_operating_kg',  'Operating weight kg'),
        ('weight_shipping_kg',   'Shipping weight kg'),
        ('dimensions_L_mm',      'Overall length mm'),
        ('dimensions_W_mm',      'Overall width mm'),
        ('dimensions_H_mm',      'Overall height mm'),
        ('oil_type',             'Lubricant type / grade'),
        ('oil_charge_L',         'Oil charge volume litres'),
    ],
    'Operating Limits': [
        ('ambient_temp_min_degC',  'Minimum ambient temperature °C'),
        ('ambient_temp_max_degC',  'Maximum ambient temperature °C'),
        ('design_pressure_kPa',   'Design pressure kPa(g)'),
        ('test_pressure_kPa',     'Test / proof pressure kPa(g)'),
        ('min_op_temp_degC',      'Minimum operating temp °C'),
        ('max_op_temp_degC',      'Maximum operating temp °C'),
    ],
    'Maintenance': [
        ('oil_change_interval_h',      'Oil change interval hours'),
        ('filter_change_interval',     'Filter replacement schedule'),
        ('overhaul_interval_h',        'Major overhaul interval hours'),
        ('alarm_high_pressure_kPa',    'High-pressure alarm setpoint kPa'),
        ('alarm_low_pressure_kPa',     'Low-pressure alarm setpoint kPa'),
        ('alarm_high_temp_degC',       'High-temperature alarm setpoint °C'),
        ('safety_cutout_pressure_kPa', 'High-pressure safety cutout kPa'),
    ],
}
print()
print('  RECOMMENDED EXTRACTION SCHEMA')
print('  (curated for industrial refrigeration equipment manuals)')
total_fields = 0
for category, fields in EXTRACTION_SCHEMA.items():
    print(f"\n  {category}  ({len(fields)} fields)")
    print(f"  {'Field Name':<35} Description")
    
    for field_name, description in fields:
        print(f"  {field_name:<35} {description}")
    total_fields += len(fields)

print(f"\n  Total fields: {total_fields}")



  RECOMMENDED EXTRACTION SCHEMA
  (curated for industrial refrigeration equipment manuals)

  Identification  (9 fields)
  Field Name                          Description
  model_number                        Model or product number of the unit
  serial_number                       Unique serial number
  item_number                         Item or tag number
  part_number                         Spare part identifier
  equipment_tag                       Plant / P&ID tag
  manufacturer                        OEM name
  document_title                      Title of the manual / datasheet
  document_revision                   Revision or version code
  document_date                       Issue or revision date

  Refrigeration Circuit  (11 fields)
  Field Name                          Description
  refrigerant_type                    Refrigerant designation e.g. R-717, R-404A
  refrigerant_charge_kg               Total refrigerant charge in kg
  compressor_type                     Screw 

---
## Summary Report — Consolidated Findings


---
## 8. AUSTCOLD — fast translation filter V2 (first 20 pages)

This test is intentionally limited to the first 20 original pages. It does **not** use OCR and renders only pages without native text; it should complete in seconds, not hours. English pages are removed only when a French counterpart is proven by document identifiers, internal page numbers, or common section-number structure.


In [15]:
# V2 imports and test scope. This cell deliberately processes ONLY 20 pages.
import re
from pathlib import Path
import pandas as pd
import pypdfium2 as pdfium
from pypdf import PdfReader, PdfWriter

AUSTCOLD_TEST_PAGES = 20
AUSTCOLD_LOOK_AROUND = 12
AUSTCOLD_TEST_OUTPUT = DATA_DIR.parent / 'filtered_pdfs' / 'AUSTCOLD_first_20_v2_filtered.pdf'
AUSTCOLD_TEST_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

FR_WORDS = set('le la les de des du et ou pour avec dans sur est sont une un au aux par ce cette ces que qui dont mais ainsi avertissement securite sécurité entretien exploitation donnees données groupe refrigeration réfrigération fiche calendrier procedure procédure soupape soupapes pression temperature température huile compresseur moteur ventilateur dessin dessins manuel manuels controle contrôle'.split())
EN_WORDS = set('the and of to in for with from this that these those is are be as by or warning safety maintenance operating operation instruction instructions package equipment service contact details standard data sheet sheets compressor compressors oil pump pumps condenser fan fans motor motors technical valve valves drawings manual manuals control controls pressure temperature schedule schedules specification specifications built routine plant warranty support information procedure procedures'.split())

def v2_language(text):
    words = re.findall(r'[A-Za-zÀ-ÿ]{2,}', text.lower())
    fr = sum(word in FR_WORDS for word in words)
    en = sum(word in EN_WORDS for word in words)
    if fr >= 3 and fr >= en * 1.3:
        return 'FR', fr, en
    if en >= 3 and en >= fr * 1.5:
        return 'EN', fr, en
    return 'UNKNOWN', fr, en

def v2_features(text):
    normal = text.upper().replace('N°', 'NO').replace('Nº', 'NO')
    document_ids = set(re.findall(r'\b[A-Z]{2,8}-\d{2,}\b', normal))
    reference_ids = set(re.findall(r'\b[A-Z]{2,}(?:[-/]?[A-Z0-9]+){2,}\b', normal))
    section_ids = set(re.findall(r'\b\d+(?:\.\d+)+\b', normal))
    page_match = re.search(r'\bPAGE\s+(\d+)\s+(?:OF|DE)\s+(\d+)\b', normal)
    logical_page = page_match.groups() if page_match else None
    return {'document_ids': document_ids, 'reference_ids': reference_ids, 'section_ids': section_ids, 'logical_page': logical_page, 'length': len(re.sub(r'\s+', ' ', text).strip())}

def v2_jaccard(left, right):
    return len(left & right) / len(left | right) if left or right else 0.0

def v2_translation_score(english, french):
    """Content evidence, never page position alone. Returns score and human-readable proof."""
    en, fr = v2_features(english), v2_features(french)
    same_doc = bool(en['document_ids'] & fr['document_ids'])
    same_logical_page = en['logical_page'] is not None and en['logical_page'] == fr['logical_page']
    section_similarity = v2_jaccard(en['section_ids'], fr['section_ids'])
    shared_refs = en['reference_ids'] & fr['reference_ids']
    length_similarity = min(en['length'], fr['length']) / max(en['length'], fr['length'], 1)

    # Strong document-internal evidence handles both manuals and bilingual tables of contents.
    if same_doc and same_logical_page:
        return 0.99, 'same document ID and same internal page number'
    if section_similarity >= 0.55 and len(en['section_ids'] & fr['section_ids']) >= 2:
        return round(0.82 + 0.15 * section_similarity, 3), f'same section-number structure ({section_similarity:.0%})'
    if same_doc and len(shared_refs) >= 2 and length_similarity >= 0.55:
        return round(0.86 + 0.10 * length_similarity, 3), 'same document ID plus shared project/reference IDs'
    return round(0.35 * section_similarity + 0.20 * length_similarity + 0.15 * float(same_doc), 3), 'insufficient bilingual-document evidence'

def v2_blank_check(pdf_path, page_index, native_text):
    """Render only no-text pages. Any visible content makes the page KEEP, never REMOVE."""
    if native_text.strip():
        return False, 'native text present'
    doc = pdfium.PdfDocument(str(pdf_path))
    try:
        image = doc[page_index].render(scale=0.30).to_pil().convert('L')
        histogram = image.histogram()
        visible_ratio = sum(histogram[:245]) / max(1, image.width * image.height)
    finally:
        doc.close()
    if visible_ratio < 0.00015:
        return True, f'visually blank (ink ratio={visible_ratio:.5f})'
    return False, f'non-text visual content retained (ink ratio={visible_ratio:.5f})'


## Language Composition of the PDF

The AUSTCOLD PDF contains bilingual content in both English and French. English pages mainly correspond to technical sections and translations of some French pages, while other pages contain content originally written in French.

The document also includes pages with little or no extractable text, which mainly contain visual content such as drawings, diagrams, or other graphical elements. Therefore, the language analysis distinguishes between French (FR), English (EN), and Unknown (UNKNOWN) pages, while also identifying some English pages as translations of corresponding French pages.

In [16]:
def run_austcold_v2_test(pdf_path=PDF_AUSTCOLD, max_pages=AUSTCOLD_TEST_PAGES):
    doc = pdfium.PdfDocument(str(pdf_path))
    try:
        context_page_count = min(max_pages + AUSTCOLD_LOOK_AROUND, len(doc))
        texts = [doc[index].get_textpage().get_text_range().strip() for index in range(context_page_count)]
    finally:
        doc.close()

    rows = []
    for index, text in enumerate(texts):
        language, fr_hits, en_hits = v2_language(text)
        is_blank, blank_reason = v2_blank_check(pdf_path, index, text)
        rows.append({
            'original_page': index + 1, 'text_length': len(text), 'language': language,
            'fr_markers': fr_hits, 'en_markers': en_hits, 'is_blank': is_blank,
            'decision': 'REMOVE' if is_blank else 'KEEP',
            'reason': 'Blank page — ' + blank_reason if is_blank else blank_reason,
            'translation_match_page': None, 'translation_score': None, 'confidence': 'HIGH' if is_blank else 'MEDIUM',
            '_text': text,
        })

    # English pages can have their French counterpart before OR after them in AUSTCOLD.
    for row_index, row in enumerate(rows):
        if row['language'] != 'EN' or row['is_blank']:
            continue
        nearby_french = [candidate_index for candidate_index in range(max(0, row_index - AUSTCOLD_LOOK_AROUND), min(len(rows), row_index + AUSTCOLD_LOOK_AROUND + 1)) if rows[candidate_index]['language'] == 'FR' and not rows[candidate_index]['is_blank']]
        candidates = [(v2_translation_score(row['_text'], rows[candidate_index]['_text']), candidate_index) for candidate_index in nearby_french]
        if not candidates:
            continue
        (score, proof), match_index = max(candidates, key=lambda item: item[0][0])
        row['translation_match_page'] = match_index + 1
        row['translation_score'] = score
        if score >= 0.82:
            row['decision'] = 'REMOVE'
            row['reason'] = f'English translation of French page {match_index + 1}: {proof}'
            row['confidence'] = 'HIGH'
        else:
            row['reason'] = f'English page retained: {proof} (score={score:.2f})'
            row['confidence'] = 'LOW'

    # Only the requested first pages are emitted/filtered; later pages are evidence only.
    return pd.DataFrame(rows[:max_pages]).drop(columns='_text')

austcold_v2_df = run_austcold_v2_test()
display(austcold_v2_df)
print('\nPages to remove:', austcold_v2_df.loc[austcold_v2_df.decision.eq('REMOVE'), 'original_page'].tolist())
print('Pages to keep  :', austcold_v2_df.loc[austcold_v2_df.decision.eq('KEEP'), 'original_page'].tolist())


,original_page,text_length,language,fr_markers,en_markers,is_blank,decision,reason,translation_match_page,translation_score,confidence
0,1,0,UNKNOWN,0,0,False,KEEP,non-text visual content retained (ink ratio=0....,NaN,NaN,MEDIUM
1,2,0,UNKNOWN,0,0,True,REMOVE,Blank page — visually blank (ink ratio=0.00000),NaN,NaN,HIGH
2,3,1026,EN,0,80,False,REMOVE,English translation of French page 6: same sec...,6.0,0.970,HIGH
3,4,913,EN,6,42,False,REMOVE,English translation of French page 7: same sec...,7.0,0.970,HIGH
4,5,376,EN,1,19,False,REMOVE,English translation of French page 8: same sec...,8.0,0.970,HIGH
5,6,1347,FR,99,3,False,KEEP,native text present,NaN,NaN,MEDIUM
6,7,1143,FR,58,1,False,KEEP,native text present,NaN,NaN,MEDIUM
7,8,423,FR,21,0,False,KEEP,native text present,NaN,NaN,MEDIUM
8,9,0,UNKNOWN,0,0,False,KEEP,non-text visual content retained (ink ratio=0....,NaN,NaN,MEDIUM
9,10,0,UNKNOWN,0,0,False,KEEP,non-text visual content retained (ink ratio=0....,NaN,NaN,MEDIUM



Pages to remove: [2, 3, 4, 5, 14, 18, 19]
Pages to keep  : [1, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 20]


In [ ]:
# Create a small review PDF from the first 20 pages only. Original AUSTCOLD.pdf is never modified.
def create_austcold_v2_test_pdf(pdf_path, decisions, output_path):
    kept_pages = decisions.loc[decisions.decision.eq('KEEP'), 'original_page'].astype(int).tolist()
    assert kept_pages == sorted(kept_pages), 'Original page order must be strictly increasing'
    reader, writer = PdfReader(str(pdf_path)), PdfWriter()
    for page_number in kept_pages:
        writer.add_page(reader.pages[page_number - 1])
    with open(output_path, 'wb') as file:
        writer.write(file)
    assert len(PdfReader(str(output_path)).pages) == len(kept_pages)
    return kept_pages

kept_test_pages = create_austcold_v2_test_pdf(PDF_AUSTCOLD, austcold_v2_df, AUSTCOLD_TEST_OUTPUT)
austcold_v2_df.to_csv(DATA_DIR.parent / 'analysis_reports' / 'AUSTCOLD_first_20_v2_decisions.csv', index=False, encoding='utf-8-sig')
print(f'Created {AUSTCOLD_TEST_OUTPUT.name} with {len(kept_test_pages)} pages, in original order.')


# AUSTCOLD.pdf --- Document structure and equipment extraction strategy

## 1. Objective of this document

This document presents the structure of the **AUSTCOLD.pdf** file, its
organization, the main types of documents it contains and the
proposed strategy for extracting the information necessary for
creating an equipment in our application.

The objective is **not** to extract all the information present
in the PDF nor to reconstruct every mechanical part.

The objective is more targeted:

> **Identify each industrial equipment and retrieve the
> global information necessary to fill in the "Create a new
> equipment" form.**

The form notably contains:

-   Family
-   Equipment name
-   Reference
-   Entity
-   Class
-   Structure
-   Group
-   Power
-   Outlier value
-   Measurement point type
-   Equipment image

------------------------------------------------------------------------

# 2. General overview of the PDF

**AUSTCOLD.pdf is an industrial technical and maintenance file**,
and not a homogeneous document.

The document groups several categories of documents:

``` text
AUSTCOLD.pdf
│
├── 1. General information / safety / history
│
├── 2. Control description and philosophy
│
├── 3. Technical data
│
├── 4. Maintenance data
│
├── 5. Equipment manuals
│
├── 6. Instrumentation manuals
│
├── 7. Valve documentation
│
└── 8. As Built Drawings
```

The document contains approximately **2,870 pages** and combines several
content formats:

-   text;
-   tables;
-   technical data sheets;
-   procedures;
-   diagrams;
-   industrial drawings;
-   manufacturer documents;
-   illustrations;
-   nameplate information;
-   parts lists;
-   control and instrumentation documents.

This heterogeneity is important: **a single identical extractor should not
be applied to all pages.**

------------------------------------------------------------------------

# 3. Organization given by the document index

The dossier index is an essential source for understanding its
organization.

## Section 1 --- Service Contact Details

This part notably contains:

-   contact details;
-   warnings;
-   safety data sheets;
-   safety precautions;
-   warranty and support information;
-   installation history;
-   oil consumption log;
-   utilities consumption;
-   cause & effect chart.

This section is mainly documentary and operational.

It is useful for the installation context but less important
for the direct identification of equipment.

------------------------------------------------------------------------

# 4. Section 2 --- Descriptions

The section mainly contains:

### 2.01 Control Philosophy

It explains the general control philosophy of the installation.

This part can be useful for understanding the relationships between
equipment, instruments, alarms and commands.

It can notably help to understand:

``` text
Equipment
    ↓
Sensor / Instrument
    ↓
Condition
    ↓
Alarm / Trip
    ↓
Control action
```

However, this section is not the main source for filling in the
general information of an equipment.

------------------------------------------------------------------------

# 5. Section 3 --- Technical Data

This section is **one of the most important for our extraction**.

It contains technical data sheets for the main equipment.

The index notably mentions:

``` text
3.01 Technical schedule
3.02 Compressor data sheet
3.03 Compressor motor data sheet
3.04 Oil pump data sheet
3.05 Oil pump motor data sheet
3.06 Oil cooler data sheet
3.07 Oil separator data sheet
3.08 Liquid receiver data sheet
3.09 Purger data sheet
3.10 Economiser data sheet
3.11 Condenser data sheet
3.12 Condenser fan data sheet
3.13 Condenser fan motor data sheet
3.14 Condenser vibration switch data sheet
3.15 Final stage oil separator data sheet
3.16 Control valve data sheets
3.17 Control and instrumentation schedules
3.18 IO list
3.19 Trip alarm setting schedules
3.20 First up alarm setting schedules
3.21 Instrument data sheets
3.22 Noise data sheet
3.23 Piping materials specification
3.24 Painting specification
```

## Why is this section important?

Technical data sheets generally make it possible to find
global information such as:

-   equipment name;
-   type;
-   manufacturer;
-   model;
-   serial number;
-   reference / tag;
-   power;
-   operating characteristics;
-   electrical data;
-   mechanical data;
-   service.

It is therefore one of the main sources for populating our
equipment record.

------------------------------------------------------------------------

# 6. Section 4 --- Maintenance Data

This part notably contains:

``` text
4.01 Maintenance schedule
4.02 Lubrication schedule
4.03 Lube oil data sheet & MSDS
4.04 Oil filter change procedure
4.05 Oil fill procedure
4.06 Commissioning spare parts list
4.07 2 year operation spare parts list
4.08 Coalescer filters change procedure
4.09 Final stage filters change procedure
```

This section is useful for:

-   identifying equipment concerned by maintenance;
-   understanding certain components;
-   finding part references;
-   understanding maintenance operations;
-   associating equipment with procedures.

But for the equipment creation form, it is secondary.

------------------------------------------------------------------------

# 7. Section 5 --- Equipment Manuals

This section contains the manufacturer manuals for the main equipment:

``` text
5.01 Refrigeration compressor — Howden
5.02 Compressor motor — Siemens
5.03 Oil pump — Viking
5.04 Oil pump motor — Weg
5.05 Oil heater — Heatex
5.06 Condenser, fan & fan motors — Jord
5.07 Purger, Economiser & Oil cooler — Flotech
```

This section is particularly interesting for:

-   confirming the equipment type;
-   confirming the manufacturer;
-   finding the model;
-   finding certain references;
-   retrieving a representative image;
-   understanding the general appearance of the equipment.

## Important

These manuals are much more detailed than what we need
for the form.

For example, a compressor manual may describe:

-   rotors;
-   bearings;
-   pistons;
-   slide valve;
-   internal components;
-   disassembly procedures;
-   spare parts.

We do not need to extract all this information to create
the equipment.

We should mainly use these manuals as a **confirmation source and image
source**.

------------------------------------------------------------------------

# 8. Section 6 --- Instrumentation Manuals

This section notably contains:

``` text
6.01 Pressure, Differential Pressure & Level Transmitters — Rosemount
6.02 Temperature Transmitter — Rosemount
```

It makes it possible to identify:

-   instruments;
-   transmitters;
-   measurement types;
-   certain references;
-   relationships between equipment and instrumentation.

It can be useful for determining the measurement context of an
equipment.

------------------------------------------------------------------------

# 9. Section 7 --- Valves

The documentation contains several types of valves:

``` text
7.01 Level control valves — Samson
7.02 Pressure relief valves — Leser
7.03 Oil differential pressure regulating valves — A.W.P.
7.04 Solenoid valve — Asco
7.05 Regulating valves — Parker
7.06 Oil temperature valves — Hansen
7.07 Hand operated manual valves — Hyundai / Supercheck
7.08 Slide Valve — see Howden
```

This section makes it possible in particular to identify valves as equipment
or sub-components depending on the application's data model.

------------------------------------------------------------------------

# 10. Section 8 --- As Built Drawings

This part is very important for understanding the **physical structure
of the installation**.

It may contain:

-   assembly drawings;
-   plans;
-   diagrams;
-   piping;
-   connections;
-   equipment;
-   instrumentation;
-   graphical representations of the installation.

## Particularity

For these pages, the text extracted from the PDF may be insufficient.

A page may contain:

``` text
a lot of graphical information
+
little text
```

Images must therefore be preserved or the page rendered as an image when
necessary.

------------------------------------------------------------------------

# 11. Main identifiable equipment

The document index already makes it possible to establish an initial list of
equipment.

### Main equipment

``` text
Compressor
Compressor Motor
Oil Pump
Oil Pump Motor
Oil Heater
Oil Cooler
Oil Separator
Final Stage Oil Separator
Liquid Receiver
Purger
Economiser
Condenser
Condenser Fan
Condenser Fan Motor
Condenser Vibration Switch
Control Valves
Instrumentation
```

This list should not be considered an exhaustive list of all objects in
the PDF.

It rather constitutes an **initial taxonomy of the equipment to be
searched for**.

------------------------------------------------------------------------

# 12. Organization by refrigeration units

The document also shows several refrigeration units.

For a given unit, an architecture composed of several equipment is found.

Conceptual example:

``` text
Refrigeration Unit A
│
├── Compressor
├── Compressor Motor
├── Oil Pump A
├── Oil Pump B
├── Oil Separator
├── Oil Filter
├── Oil Cooler
├── Liquid Receiver
├── Economiser
├── Condenser
├── Condenser Fan
├── Purger
└── Instrumentation / Valves
```

The following units may have a similar architecture.

This provides important information for extraction:

> **The entity or unit can be used as a hierarchical level
> to group equipment.**

------------------------------------------------------------------------

# 13. Recommended data hierarchy

We can represent the installation in this form:

``` text
Installation
│
├── Entity / Unit A
│   │
│   ├── Compressor
│   ├── Compressor Motor
│   ├── Oil Pump
│   ├── Oil Separator
│   ├── Oil Cooler
│   └── ...
│
├── Entity / Unit B
│   └── ...
│
├── Entity / Unit C
│   └── ...
│
└── Entity / Unit D
    └── ...
```

This hierarchy is much more useful for our application than the
physical structure of the PDF itself.

------------------------------------------------------------------------

# 14. Information to actually extract

Our application form imposes a set of fields.

## Main fields

``` text
Family
Equipment name
Reference
Entity
Class
Structure
Group
Power
Outlier value
Measurement point type
Image
```

The extraction must therefore be oriented toward these fields.

------------------------------------------------------------------------

# 15. Separating extraction and classification

This is an essential point of the architecture.

We must distinguish:

## Step 1 --- Extraction

Extract what is actually written in the PDF.

Example:

``` json
{
  "equipment_name": "Refrigeration Compressor",
  "reference": "255-V-100A-C-01",
  "manufacturer": "Howden",
  "model": "WRVi 255",
  "power": "...",
  "entity": "Refrigeration Unit A"
}
```

## Step 2 --- Classification

Transform the extracted information into the categories expected by
the application:

``` json
{
  "family": "...",
  "class": "...",
  "structure": "...",
  "group": "..."
}
```

The extractor should not be asked to directly "guess" all
the application's categories.

------------------------------------------------------------------------

# 16. Fields that do not necessarily come from the PDF

Some form fields may not be explicitly present
in the documents.

### Outlier value

The PDF may provide normal values, limits, alarms or
thresholds, but the "outlier value" field is primarily a
parameter of our application.

A business rule must therefore be defined.

### Measurement point type

The PDF contains instrumentation information, but this does
not mean that we can always directly deduce:

``` text
Manual
```

or:

``` text
Online
```

This field must be defined based on the available information and, if
necessary, a business rule.

------------------------------------------------------------------------

# 17. Image management

We should not extract all images from the PDF.

The objective is to retrieve **one representative image of
the equipment**.

For each equipment, search as a priority for:

``` text
1. General photo of the equipment
2. General view in the manual
3. General Arrangement / drawing
4. Illustration from the technical data sheet
```

Avoid choosing:

-   an electrical diagram;
-   a small icon;
-   a table converted into an image;
-   an internal view of a part;
-   an image with no direct relation to the equipment.

------------------------------------------------------------------------

# 18. Recommended data structure

A simple intermediate structure can be used:

``` json
{
  "equipment_name": "...",
  "equipment_type": "...",
  "reference": "...",
  "entity": "...",
  "manufacturer": "...",
  "model": "...",
  "serial_number": "...",
  "power": "...",

  "image": {
    "path": "...",
    "page": 123,
    "type": "equipment_view"
  },

  "source": {
    "page": 123,
    "document_section": "Technical Data",
    "document_type": "datasheet"
  }
}
```

Then a classification step can add:

``` json
{
  "family": "...",
  "class": "...",
  "structure": "...",
  "group": "..."
}
```

------------------------------------------------------------------------

# 19. Proposed pipeline

The recommended overall pipeline is:

``` text
                          AUSTCOLD.pdf
                              │
                              ▼
                     Page classification
                              │
             ┌────────────────┼────────────────┐
             ▼                ▼                ▼
       Technical Data   Equipment Manuals   Drawings
             │                │                │
             └────────────────┼────────────────┘
                              ▼
                     Equipment identification
                              │
                              ▼
                        Global extraction
                              │
             ┌─────────────────┼─────────────────┐
             ▼                 ▼                 ▼
           Text              Tables            Images
             │                 │                 │
             └─────────────────┼─────────────────┘
                              ▼
                       Equipment record
                              │
                              ▼
                         Classification
                              │
                              ▼
                            Form
```

------------------------------------------------------------------------

# 20. Page classification

Before extraction, it is preferable to determine the type of each
page.

Example:

``` text
page 001 → cover
page 002 → document_index
page 003 → safety
page 020 → technical_datasheet
page 100 → equipment_manual
page 500 → spare_parts
page 1000 → drawing
page 1500 → instrumentation
```

This then makes it possible to use an extractor adapted to the page type.

------------------------------------------------------------------------

# 21. Why not use a single extractor?

Because the formats are different.

### Technical data sheet

``` text
LABEL → VALUE
```

### Table

``` text
| Parameter | Value | Unit |
```

### Manual

``` text
Title
Paragraphs
Subsections
Figures
Tables
```

### Drawing

``` text
Text + coordinates + graphical elements
```

### Parts list

``` text
Reference → Part Number → Description → Quantity
```

A single extractor would therefore be less robust.

------------------------------------------------------------------------

# 22. Proposed technical strategy

## Level 1 --- PDF extraction

Use the tools already available in the project to:

-   extract text;
-   retrieve coordinates;
-   retrieve images;
-   identify pages;
-   detect tables when possible.

## Level 2 --- Page classification

Classify pages according to their type:

``` text
technical_datasheet
equipment_manual
maintenance
instrumentation
valves
drawing
other
```

## Level 3 --- Equipment extraction

Search for:

``` text
equipment name
tag/reference
manufacturer
model
serial number
power
entity/unit
```

## Level 4 --- Image

Associate a representative image with the equipment.

## Level 5 --- Business classification

Map the equipment to:

``` text
Family
Class
Structure
Group
```

## Level 6 --- Result generation

Produce a result directly usable by the form.

------------------------------------------------------------------------

# 23. Expected final example

For a given equipment, the system should produce something
like:

``` json
{
  "family": "Compressor",
  "equipment_name": "Refrigeration Compressor",
  "reference": "255-V-100A-C-01",
  "entity": "Refrigeration Unit A",
  "class": "...",
  "structure": "...",
  "group": "...",
  "power": "...",
  "outlier_value": null,
  "measurement_point_type": "...",
  "image": "images/compressor_255-V-100A-C-01.png"
}
```

The `...` values should only be filled in after definition of the
taxonomy and business rules of the application.

------------------------------------------------------------------------

# 24. What NOT to do

### ❌ Do not extract all mechanical parts

We do not need a detailed inventory of:

``` text
bearing
rotor
shaft
bolt
seal
washer
...
```

unless information is necessary for another need
of the application.

### ❌ Do not extract all images

We want one representative image per equipment.

### ❌ Do not process all pages in the same way

The PDF contains several document types.

### ❌ Do not invent missing fields

If information is not found:

``` json
{
  "power": null
}
```

is preferable to an invented value.

### ❌ Do not confuse extraction and classification

The PDF provides technical data.

The application has its own taxonomy.

------------------------------------------------------------------------

# 25. Recommended first development phase

Before launching extraction on the ~2,870 pages, build a prototype
on a small representative sample:

``` text
5 pages — Technical Data
10 pages — Equipment Manual
5 pages — Parts / illustrations
5 pages — Drawings
5 pages — Instrumentation / valves
```

For each page, keep:

``` text
page number
document type
extracted text
tables
images
identified equipment
reference
source
```

Then manually verify the results.

------------------------------------------------------------------------

# 26. Expected prototype result

The prototype should produce a table such as:

``` text
  Equipment         Reference   Entity   Power   Image   Source
  ------------------ ----------- -------- ------- ------- ------------------
  Compressor         ...         Unit A   ...     ✓       Technical Data
  Compressor Motor   ...         Unit A   ...     ✓       Technical Data
  Oil Pump           ...         Unit A   ...     ✓       Technical Data
  Oil Separator      ...         Unit A   ...     ✓       Technical Data
  Oil Cooler         ...         Unit A   ...     ✓       Technical Data
  Condenser          ...         Unit A   ...     ✓       Equipment Manual
```

This table constitutes the **main intermediate output** before
mapping to the form.

------------------------------------------------------------------------

# 27. Conclusion

AUSTCOLD.pdf is an **industrial technical file structured into
several document categories**, with technical data sheets,
manufacturer manuals, maintenance documents, instrumentation, valves and
as-built drawings.

For our need, the optimal strategy is not to perform an
exhaustive extraction of the document.

We need to build an extraction oriented toward **equipment**:

``` text
PDF
 ↓
Document type
 ↓
Equipment
 ↓
Global information
 ↓
Representative image
 ↓
Business classification
 ↓
Form
```

The final objective is therefore to transform:

``` text
AUSTCOLD.pdf
```

into a structured database:

``` text
Equipment
├── Name
├── Reference
├── Entity
├── Manufacturer
├── Model
├── Power
├── Image
├── Family
├── Class
├── Structure
└── Group
```

This approach is simpler, more robust and above all directly
aligned with the equipment creation form.
